# Instructions

**Please follow these instructions carefully to ensure your submission is graded correctly:**

- Fill in **your answers** in any place that says "YOUR CODE HERE" or "YOUR ANSWER HERE" in the provided `.ipynb` files.

- Do not use print statements or the variable name `_` to display your answers; instead, make sure the last line of the cell contains the variable that holds your answer.

- Your answers to code-based questions must be **generated from code**. Hardcoded answers will result in a deduction of points.  

- Do not change the **autograding cells** labelled with `"AUTOGRADING CELL - DO NOT TOUCH!"`. Modifying or deleting them may lead to incorrect grading results.

- All plots must have appropriate **axis labels** including units where applicable. Failing to do so will result in a partial deduction of points.

- Always proide **units** for quantities where applicable. Failing to do so will result in a partial deduction of points.

- The **code quality** will not be graded. However, concise, well-structured code makes it easier to awared partial points in case of errors.

- The quality of **written answers** and **figures** will be considered during grading.

- Do not use additional packages that are not part of the course **environment**. This would lead to errors during autograding.

- Before you turn this problem in, make sure everything runs as expected. First, **restart the kernel** (in the menubar, select Kernel $\rightarrow$ Restart) and then **run all cells** (in the menubar, select Cell $\rightarrow$ Run All).

- Submissions must be your own work, **plagiarism** from the web or peers will be checked for and sanctioned.

- Please upload only the provided `.ipynb` files, filled in with your answers.

- Please **do not** put the `.ipynb` files into archives (e.g., `.zip`, `.tar.gz`); also `.py` files will **not** be accepted. Please **do not change the file names** and do not upload additional files.

- Submission on ISIS *(for TU Berlin students)* requires **two-factor authentication**.

---

## Task 3: Optimisation for Electricity Market Modelling (28 points)

Build an electricity market model for minimising operational costs within technical constraints for three regions in three consecutive hours considering the following information:

The operational fleet of power plants in the three regions is specified as follows:

| Name | Technology     | Region     | Marginal Cost [€/MWh] | Capacity [MW] |
| ----- | -------------- | ---------- | --------------------- | ------------- |
| Coal A | Coal           | A | 35                    | 8000         |
| RES A| RES | A | 1                     |  4000         |
| Gas A | Gas            | A | 80                    | 4000         |
| Hydro B | Hydro          | B | 2                     | 4000          |
| Gas B | Gas            | B | 70                    | 2000          |
| RES C | RES            | C | 0                     | 5000          |

The electricity demand time series in the regions read as follows:

| Region / Time | t1 | t2 | t3 |
| --- | ---: | ---: | ---: |
| A | 10000 | 11000 | 12000 |
| B | 4000 | 4500 | 5000 |
| C | 1000 | 1200 | 1500 |

The transmission capacities (bidirectional) read as follows:

| Start      | End        | Capacity [MW] |
| ---------- | ---------- | ------------- |
| A | B | 2000           |
| B | C | 300            |
| A | C | 4000           |

The regions are close to each other, so that the RES capacity factors are the same in all three regions and read as follows:

| Time | Capacity Factor [-] |
| ---- | --------------- |
| t1 | 0.8             |
| t2 | 0.5             |
| t3 | 0.2             |

A storage unit with a maximum charge and discharge power of 2000 MW and an energy-to-power ratio of 2h is located in region A. The storage unit has a charge efficiency of 90% and a discharge efficiency of 90%. Assume that it starts with an empty state of charge.

Assume equal non-zero reactances for the transmission lines.

**Note: None of the answers can be hard-coded numbers. All answers must be derived from the model results.**

In [2]:
import linopy
import pypsa
import numpy as np
import pandas as pd

In [3]:
# indices for time steps, regions and technologies
time = pd.Index(["t1", "t2", "t3"], name="time")
regions = pd.Index(["A", "B", "C"], name="region")
techs = pd.Index(["Coal", "RES", "Gas", "Hydro"], name="technology")

# marginal costs in EUR/MWh
marginal_costs = pd.DataFrame(
    {
        "A": {"Coal": 35, "RES": 1, "Gas": 80},
        "B": {"Gas": 70, "Hydro": 2},
        "C": {"RES": 0},
    }
)
marginal_costs.columns.name = "region"
marginal_costs.index.name = "technology"


# power plant capacities (nominal powers in MW) in each country
capacities = pd.DataFrame(
    {
        "A": {"Coal": 8000, "RES": 4000, "Gas": 4000},
        "B": {"Gas": 2000, "Hydro": 4000},
        "C": {"RES": 5000},
    }
)
capacities.columns.name = "region"
capacities.index.name = "technology"

capacity_factors = pd.Series([0.8, 0.5, 0.2], index=time)

# transmission capacities in MW
transmission = pd.Series({"A-B": 2000, "A-C": 4000, "B-C": 300})
transmission.index.name = "transmission"

# country electrical loads in MW
loads = pd.DataFrame(
    {
        "A": {"t1": 10000, "t2": 11000, "t3": 12000},
        "B": {"t1": 4000, "t2": 4500, "t3": 5000},
        "C": {"t1": 1000, "t2": 1200, "t3": 1500},
    }
)
loads.columns.name = "region"
loads.index.name = "time"

**a)** Build and solve the optimisation model using `linopy` by following the steps outlined below.

**a.1)** Create a `linopy.Model` instance under the name `m`. **(1 point)**

**a.2)** Create all variables for generation (`m.g`) and transmission (`m.f`). **(1 point)**

**a.3)** Create the objective function to minimise total operational costs. **(1 point)**

**a.4)** Build the necessary constraints for generation limits, transmission limits, storage equations and the Kirchhoff laws. **(7 points)**

**a.5)** Solve the model with HiGHS. **(1 point)**

In [77]:
# YOUR CODE HERE
m = linopy.Model()
storage_power, storage_energy, efficiency = 2000, 4000, 0.9

g = m.add_variables(lower=0, upper=capacities, coords=[time, techs, regions], mask=capacities.notna(), name="g")
f = m.add_variables(
    lower=np.tile(-transmission.values, (len(time), 1)),
    upper=np.tile(transmission.values, (len(time), 1)),
    coords=[time, transmission.index],
    dims=["time", "transmission"],
    name="f"
)
charge = m.add_variables(lower=0, upper=storage_power, coords=[time], name="charge")
discharge = m.add_variables(lower=0, upper=storage_power, coords=[time], name="discharge")
soc = m.add_variables(lower=0, upper=storage_energy, coords=[time], name="soc")

m.add_objective((g * marginal_costs).sum(), sense="min")
res_max = pd.DataFrame({r: capacity_factors * capacities.loc["RES", r] for r in regions}).fillna(0)
m.add_constraints(g.sel(technology="RES") <= res_max, name="res_limit")
for i, t in enumerate(time):
    prev_soc = soc.sel(time=time[i - 1]) if i > 0 else 0
    m.add_constraints(
        soc.sel(time=t) == prev_soc + efficiency * charge.sel(time=t) - discharge.sel(time=t) / efficiency,
        name=f"soc_consistency_{t}"
    )
m.add_constraints(f >= -transmission.values, name="f_lower")
m.add_constraints(f <= transmission.values, name="f_upper")
m.add_constraints(
    f.sel(transmission="A-B") + f.sel(transmission="B-C") - f.sel(transmission="A-C") == 0,
    name="kvl"
)

gen_by_region = g.sum(dims="technology")

m.add_constraints(
    gen_by_region.sel(region="A") + discharge - charge - f.sel(transmission="A-B") - f.sel(transmission="A-C") == loads["A"],
    name="balance_A"
)
m.add_constraints(
    gen_by_region.sel(region="B") + f.sel(transmission="A-B") - f.sel(transmission="B-C") == loads["B"],
    name="balance_B"
)
m.add_constraints(
    gen_by_region.sel(region="C") + f.sel(transmission="A-C") + f.sel(transmission="B-C") == loads["C"],
    name="balance_C"
)

m.solve(solver_name="highs")

/Users/vicky/ENTER/envs/esm-ss-26/lib/python3.13/site-packages/linopy/common.py:177: UserWarning: coords for dimension(s) ['technology', 'region'] is not aligned with the pandas object. Previously, the indexes of the pandas were ignored and overwritten in these cases. Now, the pandas object's coordinates are taken considered for alignment.
  warn(


Running HiGHS 1.14.0 (git hash: n/a): Copyright (c) 2026 under MIT licence terms
LP linopy-problem-jznfa_5p has 39 rows; 36 cols; 86 nonzeros
Coefficient ranges:
  Matrix  [9e-01, 1e+00]
  Cost    [1e+00, 8e+01]
  Bound   [3e+02, 8e+03]
  RHS     [3e+02, 1e+04]
Presolving model
15 rows, 34 cols, 60 nonzeros 0s
Dependent equations search running on 12 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
12 rows, 30 cols, 50 nonzeros 0s
Presolve reductions: rows 12(-27); columns 30(-6); nonzeros 50(-36) 
Solving the presolved LP
Using dual simplex solver
  Iteration        Objective     Infeasibilities num(sum)
          0     0.0000000000e+00 Ph1: 0(0) 0.0s
         14     1.1234197531e+06 Pr: 0(0) 0.0s

Performed postsolve
Solving the original LP from the solution after postsolve

Model name          : linopy-problem-jznfa_5p
Model status        : Optimal
Simplex   iterations: 14
Objective value     :  1.1234197531e+

('ok', 'optimal')

**a.6)** Return the total operational cost as a `float`. **(1 point)**

In [79]:
# YOUR CODE HERE
cost = m.objective.value
cost

1123419.7530864198

In [80]:
### AUTOGRADING CELL - DO NOT TOUCH!
assert isinstance(_, float)


**a.7)** Return the optimal generator dispatch in region `A` as a `pandas.DataFrame` where the index represents the technology of the generator (e.g. `Coal`), the columns represent the time step (e.g. `t1`), and the values represent the optimal dispatch in MW. **(1 point)**

In [81]:
# YOUR CODE HERE
gen_df = g.solution.sel(region="A").to_pandas().T.reindex(techs, fill_value=0.0)
gen_df.index.name = "technology"
gen_df.columns.name = "time"
gen_df


time,t1,t2,t3
technology,,,
Coal,5100.0,8000.0,8000.0
RES,3200.0,2000.0,800.0
Gas,0.0,0.0,1300.0
Hydro,NaN,NaN,NaN


In [82]:
### AUTOGRADING CELL - DO NOT TOUCH!
assert isinstance(_, pd.DataFrame)
assert _.columns.symmetric_difference(["t1", "t2", "t3"]).empty
assert _.index.symmetric_difference(["Coal", "Gas", "Hydro", "RES"]).empty

**a.8)** Return the optimal power flows as a `pandas.DataFrame` where the index represents the name of the transmission line (e.g. `A-B`), the columns represent the time step (e.g. `t1`), and the values represent the directed power flow in MW. **(1 point)**

In [94]:
f_solution = f.solution.to_dataframe().reset_index()
f_pivot = f_solution.pivot_table(index='transmission', columns='time', values='solution')
f_pivot

time,t1,t2,t3
transmission,,,
A-B,-1700.0,-584.567901,-100.0
A-C,-2000.0,-884.567901,200.0
B-C,-300.0,-300.000000,300.0


In [95]:
### AUTOGRADING CELL - DO NOT TOUCH!
assert isinstance(_, pd.DataFrame)
assert _.columns.symmetric_difference(["t1", "t2", "t3"]).empty
assert _.index.symmetric_difference(["A-B", "A-C", "B-C"]).empty

**a.9)** Return the storage unit's operation as a single `pandas.Series` where the index represents the time step (e.g. `t1`) and the values represent the dispatch in MW (positive for discharging, negative for charging). **(1 point)**

In [110]:
# YOUR CODE HERE
storage_series = (discharge.solution - charge.solution).to_pandas()
storage_series


time
t1   -2000.000000
t2    -469.135802
t3    2000.000000
Name: solution, dtype: float64

In [111]:
### AUTOGRADING CELL - DO NOT TOUCH!
assert isinstance(_, pd.Series)
assert _.index.symmetric_difference(time).empty

**a.10)** Return the locational marginal prices (LMPs) as a `pandas.DataFrame` where the index represents the name of the region (e.g. `A`), and the columns represent the time step (e.g. `t1`), and the values represent the LMPs in €/MWh. **(1 point)**

In [152]:
# YOUR CODE HERE
lmps_df = pd.DataFrame({
    "A": m.constraints["balance_A"].dual.to_pandas(),
    "B": m.constraints["balance_B"].dual.to_pandas(),
    "C": m.constraints["balance_C"].dual.to_pandas()
}).T 

lmps_df.index.name = "region"
lmps_df.columns.name = "time"

lmps_df

time,t1,t2,t3
region,,,
A,35.0,35.0,80.0
B,70.0,70.0,70.0
C,-0.0,-0.0,90.0


In [153]:
### AUTOGRADING CELL - DO NOT TOUCH!
assert isinstance(_, pd.DataFrame)
assert _.columns.symmetric_difference(["t1", "t2", "t3"]).empty
assert _.index.symmetric_difference(["A", "B", "C"]).empty

**b)** Build and solve the optimisation model using `pypsa` by following the steps outlined below.

**b.1)** Create a `pypsa.Network` instance under the name `n` with the three snapshots. **(1 point)**

**b.2)** Add the loads, generators, lines, and storage unit to the network. **(5 points)**

**b.3)** Solve the network with HiGHS. **(1 point)**

In [208]:
# YOUR CODE HERE
n = pypsa.Network()
n.set_snapshots(time)
for region in regions:
    n.add("Bus", region)
    n.add("Load", f"load for {region}", bus=region, p_set=loads[region])
for region in regions:
    for tech, p_nom in capacities[region].dropna().items():
        p_max = capacity_factors if tech == "RES" else 1.0
        n.add(
            "Generator", 
            f"{tech} {region}",  # Formatted as "Coal A" for the autograder
            bus=region, 
            carrier=tech, 
            p_nom=p_nom, 
            marginal_cost=marginal_costs.loc[tech, region],
            p_max_pu=p_max
        )
for line_name, s_nom in transmission.items():
    bus0, bus1 = line_name.split("-")
    n.add("Line", line_name, bus0=bus0, bus1=bus1, s_nom=s_nom, x = 1.0)
n.add("StorageUnit", "storage", bus="A", carrier = "battery storage", max_hours=2, efficiency_store = 0.9, efficiency_dispatch = 0.9, p_nom=2000, 
    p_nom_extendable=False, state_of_charge_initial=0.0, cyclic_state_of_charge=False)
n.optimize(solver_name="highs")


/var/folders/1s/5g1n6w4x0ksckcmsmvfzf3zw0000gn/T/ipykernel_86384/852758771.py:24: FutureWarning: The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by not including the objective constant as a variable.
  n.optimize(solver_name="highs")
Index(['A', 'B', 'C'], dtype='object', name='name')
Index(['Coal A', 'RES A', 'Gas A', 'Gas B', 'Hydro B', 'RES C'], dtype='object', name='name')
Index(['storage'], dtype='object', name='name')
Index(['A-B', 'A-C', 'B-C'], dtype='object', name='name')
Index(['A-B', 'A-C', 'B-C'], dtype='object', name='name')
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io: Writing time: 0.24s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 36 primals, 87 duals
Objective: 1.12e+06
Solver model: available
Solver message: Optimal

INFO:pypsa.

Running HiGHS 1.14.0 (git hash: n/a): Copyright (c) 2026 under MIT licence terms
LP linopy-problem-o3x2owu_ has 87 rows; 36 cols; 134 nonzeros
Coefficient ranges:
  Matrix  [9e-01, 1e+05]
  Cost    [1e+00, 8e+01]
  Bound   [0e+00, 0e+00]
  RHS     [3e+02, 1e+04]
Presolving model
15 rows, 36 cols, 62 nonzeros 0s
10 rows, 30 cols, 46 nonzeros 0s
Dependent equations search running on 10 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
10 rows, 29 cols, 45 nonzeros 0s
Presolve reductions: rows 10(-77); columns 29(-7); nonzeros 45(-89) 
Solving the presolved LP
Using dual simplex solver
  Iteration        Objective     Infeasibilities num(sum)
          0     2.8079999946e+05 Pr: 7(44600) 0.0s
         12     1.1234197531e+06 Pr: 0(0) 0.0s

Performed postsolve
Solving the original LP from the solution after postsolve

Model name          : linopy-problem-o3x2owu_
Model status        : Optimal
Simplex   iterations: 12


('ok', 'optimal')

**b.4)** Return the total operational cost as a `float`. *(Hint: It should be the same as in the previous task.)* **(1 point)**

In [209]:
# YOUR CODE HERE
n.objective

1123419.7530864198

In [176]:
### AUTOGRADING CELL - DO NOT TOUCH!
assert isinstance(_, float)

**b.5)** Return the optimal generator dispatch as a `pandas.DataFrame` where the index represents the time step, the columns the name of the generator (e.g. `Coal A`), and the values the optimal dispatch in MW. **(1 point)**

In [ ]:
# YOUR CODE HERE
gen_dispatch = n.generators_t.p.copy()
gen_dispatch.drop(columns=["index", "time"], errors="ignore", inplace=True)
gen_dispatch.columns.name = "generator"
gen_dispatch.index.rename("time", inplace=True)
gen_dispatch


generator,Coal A,RES A,Gas A,Gas B,Hydro B,RES C
time,,,,,,
t1,4269.135802,3200.0,-0.0,1700.0,4000.0,3600.0
t2,8000.000000,2000.0,-0.0,900.0,4000.0,2500.0
t3,8000.000000,800.0,1300.0,1400.0,4000.0,1000.0


In [204]:
### AUTOGRADING CELL - DO NOT TOUCH!
assert isinstance(_, pd.DataFrame)
expected_columns = [tech + " " + region for (tech, region) in capacities.stack().index]
assert _.columns.symmetric_difference(expected_columns).empty

**b.6)** Return the storage unit's operation as a single `pandas.Series` where the index represents the time step (e.g. `t1`) and the values represent the dispatch in MW (positive for discharging, negative for charging). **(1 point)**

In [ ]:
# YOUR CODE HERE
n.storage_units_t.p["storage"]


snapshot
t1   -1769.135802
t2    -700.000000
t3    2000.000000
Name: storage, dtype: float64

In [239]:
### AUTOGRADING CELL - DO NOT TOUCH!
assert isinstance(_, pd.Series)
assert _.index.symmetric_difference(time).empty

**b.7)** Return the optimal power flows as a `pandas.DataFrame` where the index represents the time step, the columns the name of the transmission line (e.g. `A-B`), and the values represent the utilisation rate of the line relative to its rated capacity in percent. **(1 point)**

In [233]:
# YOUR CODE HERE
utilization_rate = (n.lines_t.p0.abs() / n.lines.s_nom * 100)
utilization_rate.index.rename("time", inplace=True)
utilization_rate


name,A-B,A-C,B-C
time,,,
t1,100.0,57.5,100.0
t2,35.0,25.0,100.0
t3,5.0,5.0,100.0


In [234]:
### AUTOGRADING CELL - DO NOT TOUCH!
assert isinstance(_, pd.DataFrame)
assert _.columns.symmetric_difference(transmission.index).empty
assert _.index.symmetric_difference(time).empty

**b.8)** Return the locational marginal prices (LMPs) as a `pandas.DataFrame` where the index represents the time step, the columns the name of the region (e.g. `A`), and the values represent the LMPs in €/MWh. **(1 point)**

In [236]:
# YOUR CODE HERE
lmp = n.buses_t.marginal_price
lmp.index.rename("time", inplace=True)
lmp


name,A,B,C
time,,,
t1,35.0,70.0,-0.0
t2,35.0,70.0,-0.0
t3,80.0,70.0,90.0


In [237]:
### AUTOGRADING CELL - DO NOT TOUCH!
assert isinstance(_, pd.DataFrame)
assert _.index.symmetric_difference(time).empty
assert _.columns.symmetric_difference(regions).empty